In [ ]:
import xgboost as xgb
import shap
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, r2_score
from joblib import Parallel, delayed
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from tqdm import tqdm
import os
from itertools import product
from sklearn.cluster import KMeans

In [ ]:
import math
import time
from sklearn.neighbors import NearestNeighbors

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 24
mpl.rcParams['axes.unicode_minus'] = 'False'

# Function define

## Adj_R2

In [ ]:
def adjusted_r2_score(y_true, y_pred, n_features):
    n_samples = len(y_true)
    r2 = r2_score(y_true, y_pred)
    
    if n_samples <= n_features + 1:
        raise ValueError(" ")
    
    adjusted_r2 = 1 - (1 - r2) * (n_samples - 1) / (n_samples - n_features - 1)
    return adjusted_r2

## RMSE

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

def calculate_metric(actual, pred, metric='RMSE'):
    if metric == 'RMSE':
        return np.sqrt(mean_squared_error(actual, pred))
    elif metric == 'MAE':
        return mean_absolute_error(actual, pred)
    elif metric == 'MAPE':
        # 避免除零，加入微小值
        eps = 1e-8
        return np.mean(np.abs((actual - pred) / (actual + eps))) * 100
    elif metric == 'SMAPE':
        denom = (np.abs(actual) + np.abs(pred)) / 2
        return np.mean(np.abs(actual - pred) / denom) * 100
    elif metric == 'NRMSE':
        rmse = np.sqrt(mean_squared_error(actual, pred))
        # 使用均值归一化
        return rmse / (np.mean(actual) + 1e-8)
    elif metric == 'R2':
        return r2_score(actual, pred)
    else:
        raise ValueError(f"Unsupported metric: {metric}")

## Plotting

In [ ]:
size = 50
def moving_wind(f1_est,X1,w=2):
    mat = np.zeros((size,size))
    for i in range(size):
        for j in range(size):
            if i == 0 and j == 0:
                top = f1_est.reshape(size,size)[0:w,0:w]
                bot = X1.reshape(size,size)[0:w,0:w]
                
            if i == 0 and j > 0:
                top = f1_est.reshape(size,size)[0:i+w,j-1:j+w]
                bot = X1.reshape(size,size)[0:i+w,j-1:j+w]
                
            if j == 0 and i > 0:
                top = f1_est.reshape(size,size)[i-1:i+w,0:j+w]
                bot = X1.reshape(size,size)[i-1:i+w,0:j+w]
                
            if i > 0 and j > 0 : 
                top = f1_est.reshape(size,size)[i-1:i+w,j-1:j+w]
                bot = X1.reshape(size,size)[i-1:i+w,j-1:j+w]
            
            m = np.linalg.lstsq(bot.reshape(-1,1), top.reshape(-1,1), rcond=None)[0][0,0]
            if m > size:
                print(i,j)
            mat[i,j] = m
            
    return mat

In [ ]:
def calculate_tikhonov_local_coefficients(shap_arr, x_arr, alpha=0.01):
    """
    Parameters:
    - shap_arr: array-like, shape (n_samples,) 
    - x_arr:    array-like, shape (n_samples,)  
    - alpha:    lambda = alpha * Var(X)。

    Return:
    - coef_arr: np.ndarray, shape (n_samples,) 
    """
    y_vals = np.asarray(shap_arr)
    x_vals = np.asarray(x_arr)
    var_x = np.var(x_vals)
    lambda_val = alpha * var_x if var_x > 0 else 1e-8

    robust_coef = (y_vals * x_vals) / (x_vals**2 + lambda_val)

    return robust_coef

In [ ]:
def plot_1(b,title,vmin=None,vmax=None):
    size = 50
    plt.figure(figsize=(6, 4), dpi=300)
    plt.tight_layout()
    plt.imshow(b.reshape(size,size),vmin=vmin, vmax=vmax)
    plt.title(title,fontsize=25)
    plt.colorbar()
    plt.xticks([])
    plt.yticks([])

## define the files save path

In [ ]:
## figure
simu_path = r'C:\\' ## Replace with your figure storage path
result_path = r'C:\\' ## Replace with your results storage path

"""
We did not fit HMGWX, MGWX, MGX,GWXGB, and GXGB in the notebook for more stable training process
The codes associated with these models in the notebook are mainly for visualization

"""


# Data Generation

In [ ]:
seed = 123456789 # please replace it with any number

np.random.seed(seed)
size = 50

X1 = np.random.uniform(-1.5,1.5,size*size)
X2 = np.random.uniform(-1.5,1.5,size*size)
X3 = np.random.uniform(-1.5,1.5,size*size)
X4 = np.random.uniform(-1.5,1.5,size*size)
X5 = np.random.uniform(-1.5,1.5,size*size)
X6 = np.random.uniform(-1.5,1.5,size*size)


X = np.vstack([X1, X2, X3, X4, X5, X6]).T

b6 = np.zeros((size,size))
b3 = np.zeros((size,size))
b4 = np.zeros((size,size))

for i in range(size):
    for j in range(size):
        b6[i][j] = 2

for i in range(size):
    for j in range(size):
        b4[i][j] = math.cos(math.pi*math.exp(i/50))*math.sin(math.pi*math.exp(j/50))*3

for i in range(size):
    for j in range(size):
        b3[i][j] = 3*((i+j)/99)

u = np.array([np.linspace(0,size-1,num=size)]*size).reshape(-1)
v = np.array([np.linspace(0,size-1,num=size)]*size).T.reshape(-1)
coords = np.array(list(zip(u,v)))

# 对角区域使用相同函数形式
f1 = np.zeros_like(X1)
# 创建区域掩码
center_u = size / 2
center_v = size / 2

# 划分四个象限
mask_top_left = (u < center_u) & (v < center_v)      # 左上
mask_top_right = (u >= center_u) & (v < center_v)    # 右上  
mask_bottom_left = (u < center_u) & (v >= center_v)  # 左下
mask_bottom_right = (u >= center_u) & (v >= center_v) # 右下

f1[mask_top_left] = np.tanh(X1[mask_top_left]*np.pi)
f1[mask_bottom_right] = np.sin(X1[mask_bottom_right] * np.pi)


f1[mask_top_right] = np.abs(X1[mask_top_right]) * X1[mask_top_right]
f1[mask_bottom_left] = X1[mask_bottom_left]**3

f2 = np.zeros_like(X2)
mask_left = u < (size / 2)
mask_right = ~mask_left
f2[mask_left] = np.abs(X2[mask_left])*X2[mask_left]
f2[mask_right] = np.tanh(X2[mask_right]*np.pi)
f3 = b3.reshape(-1)*X3.reshape(-1)
f4 = b4.reshape(-1)*X4.reshape(-1)
f5 = X5**3
f6 = 2*X6

fs = np.vstack([f1, f2, f3, f4, f5, f6]).T
err = np.random.uniform(-1.5,1.5,size*size)
y = (f1 + f2 + f3 + f4 + f5 + f6 + err).reshape(-1,1)

In [ ]:
def generate_dataset(seed, size=50):
    np.random.seed(seed)
    
    X1 = np.random.uniform(-1.5, 1.5, size*size)
    X2 = np.random.uniform(-1.5, 1.5, size*size)
    X3 = np.random.uniform(-1.5, 1.5, size*size)
    X4 = np.random.uniform(-1.5, 1.5, size*size)
    X5 = np.random.uniform(-1.5, 1.5, size*size)
    X6 = np.random.uniform(-1.5, 1.5, size*size)

    X = np.vstack([X1, X2, X3, X4, X5, X6]).T

    b6 = np.zeros((size, size))
    b3 = np.zeros((size, size))
    b4 = np.zeros((size, size))

    for i in range(size):
        for j in range(size):
            b6[i][j] = 2
            b4[i][j] = math.cos(math.pi*math.exp(i/50))*math.sin(math.pi*math.exp(j/50))*3
            b3[i][j] = 3*((i+j)/99)

    u = np.array([np.linspace(0, size-1, num=size)]*size).reshape(-1)
    v = np.array([np.linspace(0, size-1, num=size)]*size).T.reshape(-1)
    coords = np.array(list(zip(u, v)))

    f1 = np.zeros_like(X1)
    center_u = size / 2
    center_v = size / 2

    mask_top_left = (u < center_u) & (v < center_v)
    mask_top_right = (u >= center_u) & (v < center_v)
    mask_bottom_left = (u < center_u) & (v >= center_v)
    mask_bottom_right = (u >= center_u) & (v >= center_v)

    f1[mask_top_left] = np.tanh(X1[mask_top_left]*np.pi)
    f1[mask_bottom_right] = np.sin(X1[mask_bottom_right] * np.pi)
    f1[mask_top_right] = np.abs(X1[mask_top_right]) * X1[mask_top_right]
    f1[mask_bottom_left] = X1[mask_bottom_left]**3

    f2 = np.zeros_like(X2)
    mask_left = u < (size / 2)
    mask_right = ~mask_left
    f2[mask_left] = np.abs(X2[mask_left])*X2[mask_left]
    f2[mask_right] = np.tanh(X2[mask_right]*np.pi)

    f3 = b3.reshape(-1)*X3.reshape(-1)
    f4 = b4.reshape(-1)*X4.reshape(-1)
    f5 = X5**3
    f6 = 2*X6

    fs = np.vstack([f1, f2, f3, f4, f5, f6]).T
    err = np.random.uniform(-1.5, 1.5, size*size)
    y = (f1 + f2 + f3 + f4 + f5 + f6 + err).reshape(-1, 1)
    
    return X, y, coords, fs

## parameter surface

In [ ]:
for i in range(6):

    fig = plt.figure(figsize = (8, 8))

    plot_1(fs[:, i] / X[:, i], fr'actual: $\beta_{i+1} $')
    
    plt.savefig(simu_path + '\\' + fr'actual_coef{i+1}_s.jpg', 
                dpi = 300)

    plt.close()

In [ ]:
for i in range(6):
    print(np.min(fs[:, i] / X[:, i]))
    print(np.max(fs[:, i] / X[:, i]))

In [ ]:
vmins = [-0.682457051890915, 3.189114018598005e-05, 0.0, -2.9987223024571503, 7.360470597936544e-08, 1.8]
vmaxs = [3.14158952787694, 3.141592563325639, 2.997501561632397, 2.9990105934556786, 2.2489935724983514, 2.2]


## non-linear relationship

In [ ]:
for i in range(6):
    
    plt.figure(figsize=(11, 9))
    
    plt.scatter(X[:, i], 
                fs[:, i], 
                c = 'k', 
                s = 3, 
                alpha = 0.7)
    
    plt.ylabel(rf'$f_{i+1} X_{i+1} \sim X_{i+1}$')
    plt.xlabel(fr'$\mathrm{{X}}_{i+1}$')
    
    plt.savefig(simu_path + '\\' + rf'true_f{i+1}_nl.jpg',
                dpi = 300)

    plt.close()

# MGWR

In [ ]:
from mgwr.gwr import GWR, MGWR                                                   
from mgwr.sel_bw import Sel_BW
import time

## Model evaluation

In [ ]:
import time
start_time = time.time()

sel = Sel_BW(coords, y, X, multi=True, constant=True, kernel='bisquare', fixed=False)
bws = sel.search(verbose=False)
result = MGWR(coords, y, X, selector=sel,constant=True, kernel='bisquare', fixed=False).fit()

end_time = time.time()
print(end_time - start_time)

In [ ]:
bws

In [ ]:
print("RMSE: ", np.sqrt(mean_squared_error(y, result.predy)))
print("adj_R2: ", adjusted_r2_score(y, result.predy, 6))

## Compute confidence level

In [ ]:
from tqdm import tqdm

def boostrap_shap(y_pred):
    n =2500
    shap_bootstrap_list = []
    err = y - y_pred
    bws_list = []
    
    for i in tqdm(range(100)):
        
        random_sample_index = np.random.choice(np.arange(2500), size=2500, replace=True)
        
        y_sample = y_pred + err[random_sample_index]
        print(y_sample.shape)
        
        sel = Sel_BW(coords, y_sample, X, multi=True, constant=True, kernel='bisquare', fixed=False)
        bws = sel.search(verbose=False)
        result = MGWR(coords, y, X, selector=sel,constant=True, kernel='bisquare', fixed=False).fit()

        shap_bootstrap_list.append(result.params)
        bws_list.append(bws)
        
    return np.concatenate(shap_bootstrap_list, axis=0), bws_list

In [ ]:
mgwr_boostrap, bws_sel = boostrap_shap(result.predy)

In [ ]:
params_bootstrap_array = mgwr_boostrap.reshape(10, 2500, 7)

params_ci_lower_simple = np.percentile(params_bootstrap_array, 2.5, axis=0)
params_ci_upper_simple = np.percentile(params_bootstrap_array, 97.5, axis=0)

## Plotting

In [ ]:
params = result.params

In [ ]:
params.shape

### Non-linear

In [ ]:
for i in range(6):
    plt.figure(figsize=(10, 7))
    
    # 为了绘制平滑的置信区间，需要按 X 值排序
    # 获取当前特征的数据
    x_vals = X[:, i]
    pred_vals = params[:, i+1] * X[:, i]
    true_vals = fs[:, i]
    lower_vals = params_ci_lower_simple[:, i+1] * X[:, i] # 假设您有对应的 lower 数组
    upper_vals = params_ci_upper_simple[:, i+1] * X[:, i]# 假设您有对应的 upper 数组
    
    # 按 X 值排序（确保置信区间绘制正确）
    sort_idx = np.argsort(x_vals)
    x_sorted = x_vals[sort_idx]
    pred_sorted = pred_vals[sort_idx]
    lower_sorted = lower_vals[sort_idx]
    upper_sorted = upper_vals[sort_idx]
    
    # 绘制置信区间（半透明填充）
    plt.fill_between(x_sorted, lower_sorted, upper_sorted, 
                     color='lightgray', alpha=0.7, label='95% Confidence Interval')
    
    # 绘制预测值散点
    plt.scatter(x_vals, pred_vals,
                c='none', edgecolors='firebrick',
                alpha=0.6, s=18, label='MGWR curve')
    
    # 绘制真实值散点
    plt.scatter(x_vals, true_vals, 
                c='k', s=2.5, alpha=0.9, label='Actual curve')
    
    # 可选：绘制预测值的连线（显示趋势）
    # plt.plot(x_sorted, pred_sorted, color='firebrick', alpha=0.4, linewidth=1)
    
    # 计算并显示指标
    metric_val = calculate_metric(pred_vals, true_vals, chosen_metric)
    plt.text(locs_X[i], locs_Y[i], f'{chosen_metric}: {metric_val:.3f}',
             fontdict={'size': 24})
    
    plt.legend(loc=locs[i], fontsize = 22)
    plt.ylabel(fr'$\phi_{i+1} (X_{i+1})$')
    plt.xlabel(fr'$\mathrm{{X}}_{i+1}$')
    
    plt.savefig(simu_path + '\\' + fr'mgwr_X{i+1}_nl.jpg', dpi=300, bbox_inches='tight')
    plt.close()

### spatial

In [ ]:
for i in range(6):

    fig = plt.figure(figsize = (8, 8))

    plot_1(result.params[:, i+1], fr'MGWR: $f_{i+1} $', vmin = vmins[i], vmax = vmaxs[i])
    
    expected = fs[:, i] / X[:, i].reshape(-1)
    predicted = result.params[:, i+1].reshape(-1)
    cosine_sim = np.dot(predicted, expected)/(np.linalg.norm(predicted) * np.linalg.norm(expected))
    
    print(f'the cosine similarity index for x_{i+1} is {cosine_sim}')

    plt.savefig(simu_path + '\\' + fr'mgwr_X{i+1}_s.jpg', 
                dpi = 300)

    plt.close()

# GGP-GAM

In [ ]:
ggp_results = pd.read_csv(result_path + '\\' + r'ggp_gam_svc_all_location_results.csv') ## generated by R code

ggp_results.head(5)

In [ ]:
ggp_ci = pd.read_csv(result_path + '\\' + r'ggp_gam_svc_all_bootstrap_ci_results.csv') ## generated by R code
ggp_ci.head()

## Model evaluation

In [ ]:
y_pred = np.array(ggp_results['predicted_y_svc_all']).reshape(-1, 1)
y_true = np.array(ggp_results['y']).reshape(-1, 1)


print("RMSE: ", np.sqrt(mean_squared_error(y_true, y_pred)))
print("adj_R2: ", adjusted_r2_score(y_true, y_pred, 6))

## plotting

In [ ]:
betas = np.array(ggp_results[['estimated_beta_X1', 'estimated_beta_X2', 'estimated_beta_X3', 'estimated_beta_X4', 'estimated_beta_X5', 'estimated_beta_X6']]).reshape(2500, 6)
params_ci_lower_simple = np.array(ggp_ci[['estimated_beta_X1_ci_lower', 'estimated_beta_X2_ci_lower', 'estimated_beta_X3_ci_lower', 'estimated_beta_X4_ci_lower', 'estimated_beta_X5_ci_lower', 'estimated_beta_X6_ci_lower']]).reshape(2500, 6)
params_ci_upper_simple = np.array(ggp_ci[['estimated_beta_X1_ci_upper', 'estimated_beta_X2_ci_upper', 'estimated_beta_X3_ci_upper', 'estimated_beta_X4_ci_upper', 'estimated_beta_X5_ci_upper', 'estimated_beta_X6_ci_upper']]).reshape(2500, 6)


### Non-linear

In [ ]:
for i in range(6):
    plt.figure(figsize=(10, 7))
    
    # 为了绘制平滑的置信区间，需要按 X 值排序
    # 获取当前特征的数据
    x_vals = X[:, i]
    pred_vals = betas[:, i] * X[:, i]
    true_vals = fs[:, i]
    lower_vals = params_ci_lower_simple[:, i] * X[:, i] # 假设您有对应的 lower 数组
    upper_vals = params_ci_upper_simple[:, i] * X[:, i]# 假设您有对应的 upper 数组
    
    # 按 X 值排序（确保置信区间绘制正确）
    sort_idx = np.argsort(x_vals)
    x_sorted = x_vals[sort_idx]
    pred_sorted = pred_vals[sort_idx]
    lower_sorted = lower_vals[sort_idx]
    upper_sorted = upper_vals[sort_idx]
    
    # 绘制置信区间（半透明填充）
    plt.fill_between(x_sorted, lower_sorted, upper_sorted, 
                     color='lightgray', alpha=0.7, label='95% Confidence Interval')
    
    # 绘制预测值散点
    plt.scatter(x_vals, pred_vals,
                c='none', edgecolors='firebrick',
                alpha=0.6, s=18, label='GGP-GAM curve')
    
    # 绘制真实值散点
    plt.scatter(x_vals, true_vals, 
                c='k', s=2.5, alpha=0.9, label='Actual curve')
    
    # 可选：绘制预测值的连线（显示趋势）
    # plt.plot(x_sorted, pred_sorted, color='firebrick', alpha=0.4, linewidth=1)
    
    # 计算并显示指标
    metric_val = calculate_metric(pred_vals, true_vals, chosen_metric)
    plt.text(locs_X[i], locs_Y[i], f'{chosen_metric}: {metric_val:.3f}',
             fontdict={'size': 24})
    
    plt.legend(loc=locs[i], fontsize = 22)
    plt.ylabel(fr'$\phi_{i+1} (X_{i+1})$')
    plt.xlabel(fr'$\mathrm{{X}}_{i+1}$')
    
    plt.savefig(simu_path + '\\' + fr'ggp_X{i+1}_nl.jpg', dpi=300, bbox_inches='tight')
    plt.close()

### spatial

In [ ]:
for i in range(6):

    fig = plt.figure(figsize = (8, 8))

    plot_1(betas[:, i], fr'GGP-GAM: $f_{i+1} $', vmin = vmins[i], vmax = vmaxs[i])
    
    expected = fs[:, i] / X[:, i].reshape(-1)
    predicted = betas[:, i].reshape(-1)
    cosine_sim = np.dot(predicted, expected)/(np.linalg.norm(predicted) * np.linalg.norm(expected))
    
    print(f'the cosine similarity index for x_{i+1} is {cosine_sim}')

    plt.savefig(simu_path + '\\' + fr'ggp_X{i+1}_s.jpg', 
                dpi = 300)

    plt.close()

# Geo-XGB

In [ ]:
from geoxgboost import create_param_grid, global_xgb, gxgb, nestedCV, optimize_bw, predict_gxgb
import shap

##  Model training

In [ ]:
Param1='n_estimators'
Param1_Values = [100, 200, 300, 500]
Param2='learning_rate'
Param2_Values = [0.1, 0.05,0.01]
Param3='max_depth'
Param3_Values = [2,3,4,6]

hyper_grid = create_param_grid(Param1,Param1_Values,Param2,Param2_Values,Param3,Param3_Values)

best_params, nestedcv_output = nestedCV(X, y, hyper_grid,
                                Param1=Param1,
                                Param2=Param2,
                                Param3=Param3,)
print("best params:", best_params)

In [ ]:
bw_min = 150   
bw_max = 1000 
step = 50

optimal_bw = optimize_bw(X, y, coords, params=best_params,
                         bw_min=bw_min, bw_max=bw_max, step=step,
                         Kernel='Adaptive', spatial_weights=True,
                         n_splits=3, path_save=False)

print("best bandwidth:", optimal_bw)

In [ ]:
%%capture

gxgb_output = gxgb(X, y, coords,
                   params=best_params,
                   bw=optimal_bw,
                   Kernel='Adaptive',
                   spatial_weights=True,  
                   feat_importance='gain',
                   alpha_wt_type='varying',
                   alpha_wt=1,       
                   test_size=0.30,
                   seed=7,
                   n_splits=5,
                   path_save=False)

In [ ]:
local_models = gxgb_output['bestLocalModel']
global_models = gxgb_output['bestGlobalModel']
alpha_wts = gxgb_output['alpha_wt']         
y_ensemble = gxgb_output['Prediction']['y_ensemble'] 

n_points = len(local_models)
n_features = X.shape[1]
feature_names = X.columns.tolist()


local_shap_df = pd.DataFrame(index=range(n_points), columns=feature_names)
global_shap_df = pd.DataFrame(index=range(n_points), columns=feature_names)
combined_shap_df = pd.DataFrame(index=range(n_points), columns=feature_names)

local_base = []  
global_base = [] 
combined_base = []  

for i in range(n_points):
    x_i = X.iloc[i:i+1, :]

    # ----- local shap -----
    explainer_local = shap.TreeExplainer(local_models[i])
    shap_local_i = explainer_local.shap_values(x_i)[0]         
    local_shap_df.iloc[i, :] = shap_local_i
    local_base.append(explainer_local.expected_value)

    # ----- global shap -----
    explainer_global = shap.TreeExplainer(global_models[i])
    shap_global_i = explainer_global.shap_values(x_i)[0]
    global_shap_df.iloc[i, :] = shap_global_i
    global_base.append(explainer_global.expected_value)

    # ----- combine shap values -----
    alpha = alpha_wts[i]
    combined_shap_i = alpha * shap_local_i + (1 - alpha) * shap_global_i
    combined_shap_df.iloc[i, :] = combined_shap_i
    combined_base_i = alpha * local_base[-1] + (1 - alpha) * global_base[-1]
    combined_base.append(combined_base_i)

local_shap_df['base_value'] = local_base
local_shap_df['pred_local'] = gxgb_output['Prediction']['LM_yOOB'].values
local_shap_df['alpha_wt'] = alpha_wts

global_shap_df['base_value'] = global_base
global_shap_df['pred_global'] = gxgb_output['Prediction']['yGhat'].values

combined_shap_df['base_value'] = combined_base
combined_shap_df['pred_ensemble'] = y_ensemble.values
combined_shap_df['alpha_wt'] = alpha_wts


In [ ]:
g_xgb_results = combined_shap_df.copy()

In [ ]:
g_xgb_results[['x1', 'x2', 'x3', 'x4', 'x5', 'x6']] = X

In [ ]:
y = np.array(g_xgb_results['y']).reshape(-1, 1)
y_hat = np.array(g_xgb_results['y_ensemble']).reshape(-1, 1)
X = np.array(g_xgb_results[['x1', 'x2', 'x3', 'x4', 'x5', 'x6']]).reshape(2500, 6)
# fs = np.array(g_xgb_results[['f1_true', 'f2_true', 'f3_true', 'f4_true', 'f5_true', 'f6_true']]).reshape(2500, 6)
fs_hat = np.array(g_xgb_results[['shap_x1', 'shap_x2', 'shap_x3', 'shap_x4', 'shap_x5', 'shap_x6']]).reshape(2500, 6)

## Evaluation

In [ ]:
y_pred = np.array(g_xgb_results['y_ensemble']).reshape(-1, 1)
y_true = np.array(g_xgb_results['y']).reshape(-1, 1)


print("RMSE: ", np.sqrt(mean_squared_error(y_true, y_pred)))
print("adj_R2: ", adjusted_r2_score(y_true, y_pred, 6))

## Plotting

### Non-linear

In [ ]:
## global relationships
locs_X = [0.5, 0.5, 0.5, -0.5, 0.5, 0.5]
locs_Y = [-3.2, -2, -4, -4, -3, -3]
locs = ['upper left', 'upper left', 'upper center','upper center', 'upper left', 'upper left']

for i in range(6):
    plt.figure(figsize=(11, 8))

    plt.scatter(X[:, i], 
                fs_hat[:, i],
                c = 'none',
                edgecolors = 'firebrick',
                alpha=0.6, 
                s = 18)
    
    plt.scatter(X[:, i], 
                fs[:, i], 
                c = 'k', 
                s = 2.5, 
                alpha = 0.7)
    
    metric_val = calculate_metric(fs_hat[:, i], fs[:, i], chosen_metric)
    plt.text(locs_X[i], locs_Y[i], f'{chosen_metric}: {metric_val:.3f}',
             fontdict={'size': 28})

    plt.legend(['G-XGBoost curve', 'Actual curve'], loc = locs[i])

    plt.ylabel(fr'$\phi_{i+1} (X_{i+1})$')
    plt.xlabel(fr'$\mathrm{{X}}_{i+1}$')

    plt.savefig(simu_path + '\\' + fr'g_xgb_X{i+1}_nl.jpg', 
                dpi = 300)

    plt.close()    

In [ ]:
chosen_metric = 'RMSE'

for i in range(6):
    plt.figure(figsize=(10, 7))
    
    # 为了绘制平滑的置信区间，需要按 X 值排序
    # 获取当前特征的数据
    x_vals = X[:, i]
    pred_vals = fs_hat[:, i]
    true_vals = fs[:, i]
    lower_vals = fs_lower[:, i]  # 假设您有对应的 lower 数组
    upper_vals = fs_upper[:, i]  # 假设您有对应的 upper 数组
    
    # 按 X 值排序（确保置信区间绘制正确）
    sort_idx = np.argsort(x_vals)
    x_sorted = x_vals[sort_idx]
    pred_sorted = pred_vals[sort_idx]
    lower_sorted = lower_vals[sort_idx]
    upper_sorted = upper_vals[sort_idx]
    
    # 绘制置信区间（半透明填充）
    plt.fill_between(x_sorted, lower_sorted, upper_sorted, 
                     color='k', alpha=0.2, label='95% Confidence Interval')
    
    # 绘制预测值散点
    plt.scatter(x_vals, pred_vals,
                c='none', edgecolors='firebrick',
                alpha=0.6, s=18, label='G-XGBoost curve')
    
    # 绘制真实值散点
    plt.scatter(x_vals, true_vals, 
                c='k', s=2.5, alpha=0.9, label='Actual curve')
    
    # 可选：绘制预测值的连线（显示趋势）
    # plt.plot(x_sorted, pred_sorted, color='firebrick', alpha=0.4, linewidth=1)
    
    # 计算并显示指标
    metric_val = calculate_metric(pred_vals, true_vals, chosen_metric)
    plt.text(locs_X[i], locs_Y[i], f'{chosen_metric}: {metric_val:.3f}',
             fontdict={'size': 24})
    
    plt.legend(loc=locs[i], fontsize = 20)
    plt.ylabel(fr'$\phi_{i+1} (X_{i+1})$')
    plt.xlabel(fr'$\mathrm{{X}}_{i+1}$')
    
    plt.savefig(simu_path + '\\' + fr'g_xgb_X{i+1}_nl.jpg', dpi=300, bbox_inches='tight')
    plt.close()

### Spatial

In [ ]:
for i in range(6):

    fig = plt.figure(figsize = (8, 8))

    pesudo = calculate_tikhonov_local_coefficients(fs_hat[:, i], X[:, i])
    
    plot_1(pesudo, fr'G-XGBoost: $f_{i+1} $', vmin = vmins[i], vmax = vmaxs[i])
    
    expected = fs[:, i] / X[:, i].reshape(-1)
    cosine_sim = np.dot(pesudo, expected)/(np.linalg.norm(pesudo) * np.linalg.norm(expected))
    
    print(f'the cosine similarity index for x_{i+1} is {cosine_sim}')

    plt.savefig(simu_path + '\\' + fr'g_xgb_X{i+1}_s.jpg', 
                dpi = 300)

    plt.close()

# XGBoost

## Model specification

In [ ]:
class XGBoostTrainer_1:
    def __init__(self, n_splits=5, random_state=42, n_jobs=-1):
        self.best_model = None
        self.best_params = None
        self.oof_predictions = None
        self.cv_scores = []
        self.n_splits = n_splits
        self.random_state = random_state
        self.best_iterations_per_fold = []
        self.oof_shap_values = None
        self.n_jobs = n_jobs

    def _train_fold(self, params, X, y, train_idx, val_idx, fold_num):

        X_train_fold, X_val_fold = X[train_idx], X[val_idx]
        y_train_fold, y_val_fold = y[train_idx], y[val_idx]

        dtrain_fold = xgb.DMatrix(X_train_fold, label=y_train_fold)
        dvalid_fold = xgb.DMatrix(X_val_fold, label=y_val_fold)

        watchlist = [(dvalid_fold, 'eval')]

        model = xgb.train(params,
                         dtrain_fold,
                         num_boost_round=1000,
                         evals=watchlist,
                         early_stopping_rounds=30,
                         verbose_eval=False)

        return {
            'fold_num': fold_num,
            'score': model.best_score,
            'model': model,
            'val_idx': val_idx
        }

    def objective(self, params, X, y):
        params['max_depth'] = int(params['max_depth'])
        params['min_child_weight'] = int(params['min_child_weight'])

        param_use = {
            'objective': 'reg:squarederror',
            'eval_metric': 'rmse',
            **params
        }

        kf = KFold(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)
        folds = list(kf.split(X, y))

        results = Parallel(n_jobs=self.n_jobs)(
            delayed(self._train_fold)(
                param_use, X, y, train_idx, val_idx, fold_num
            )
            for fold_num, (train_idx, val_idx) in enumerate(folds)
        )

        cv_scores = [res['score'] for res in results]
        avg_cv_score = np.mean(cv_scores)
        
        return {'loss': avg_cv_score, 'status': STATUS_OK}

    def _cv_fold(self, X, y, train_idx, val_idx, fold_num):
        X_train_fold, X_val_fold = X[train_idx], X[val_idx]
        y_train_fold, y_val_fold = y[train_idx], y[val_idx]

        dtrain_fold = xgb.DMatrix(X_train_fold, label=y_train_fold)
        dvalid_fold = xgb.DMatrix(X_val_fold, label=y_val_fold)

        watchlist = [(dtrain_fold, 'train'), (dvalid_fold, 'eval')]

        fold_model = xgb.train(self.best_params,
                             dtrain_fold,
                             num_boost_round=2000,
                             evals=watchlist,
                             early_stopping_rounds=50,
                             verbose_eval=100 if fold_num == 0 else False)

        best_iteration = fold_model.best_iteration
        fold_score = fold_model.best_score

        oof_preds_fold = fold_model.predict(dvalid_fold, iteration_range=(0, best_iteration))
        
        fold_explainer = shap.TreeExplainer(fold_model)
        fold_shap_value = fold_explainer.shap_values(dvalid_fold)

        return {
            'fold_num': fold_num,
            'best_iteration': best_iteration,
            'score': fold_score,
            'val_idx': val_idx,
            'oof_pred': oof_preds_fold,
            'shap_values': fold_shap_value,
            'model': fold_model
        }

    def calcu_oof_and_shap(self, X, y, tune=True, max_evals=50):
        if y.ndim > 1 and y.shape[1] == 1:
            y = y.ravel()
            
        if X.ndim == 1:
            X = X.reshape(-1, 1)

        if tune or self.best_params is None:
            self.tune_params(X, y, max_evals=max_evals)
        elif not tune and self.best_params is None:
            raise ValueError("Cannot train without tuning if best_params are not already set.")

        print(f"\nStarting {self.n_splits}-Fold CV for OOF predictions (parallel)...")
        kf = KFold(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)
        folds = list(kf.split(X, y))

        self.oof_predictions = np.zeros(X.shape[0])
        self.oof_shap_values = np.zeros(X.shape)
        self.cv_scores = []
        self.best_iterations_per_fold = []

        results = Parallel(n_jobs=self.n_jobs)(
            delayed(self._cv_fold)(X, y, train_idx, val_idx, fold_num)
            for fold_num, (train_idx, val_idx) in enumerate(folds)
        )

        for res in results:
            self.best_iterations_per_fold.append(res['best_iteration'])
            self.cv_scores.append(res['score'])
            self.oof_predictions[res['val_idx']] = res['oof_pred']
            self.oof_shap_values[res['val_idx']] = res['shap_values']

        oof_rmse = np.sqrt(mean_squared_error(y, self.oof_predictions))
        oof_r2 = adjusted_r2_score(y, self.oof_predictions, X.shape[1])
        
        print("\n--- Training Final Model on Full Data ---")
        final_num_boost_round = int(np.median(self.best_iterations_per_fold))
        dfull = xgb.DMatrix(X, label=y)

        self.best_model = xgb.train(self.best_params,
                                  dfull,
                                  num_boost_round=final_num_boost_round,
                                  verbose_eval=100)

        print("Final model training complete.")
        return self.best_model, self.oof_predictions, oof_rmse, oof_r2, self.oof_shap_values

    def tune_params(self, X, y, max_evals=50):
        search_space = {
            "max_depth": hp.quniform('max_depth', 3, 8, 1),
            "learning_rate": hp.loguniform('learning_rate', np.log(0.01), np.log(0.2)),
            "subsample": hp.uniform('subsample', 0.6, 1.0),
            "colsample_bytree": hp.uniform('colsample_bytree', 0.6, 1.0),
            "min_child_weight": hp.quniform('min_child_weight', 1, 6, 1),
            "gamma": hp.uniform('gamma', 0, 0.5),
            "reg_alpha": hp.loguniform('reg_alpha', np.log(0.001), np.log(1.0)),
            "reg_lambda": hp.loguniform('reg_lambda', np.log(0.1), np.log(10.0)),
        }

        trials = Trials()
        best = fmin(
            fn=lambda params: self.objective(params, X, y),
            space=search_space,
            algo=tpe.suggest,
            max_evals=max_evals,
            trials=trials,
            rstate=np.random.default_rng(self.random_state)
        )

        self.best_params = {
            'max_depth': int(best['max_depth']),
            'learning_rate': best['learning_rate'],
            'subsample': best['subsample'],
            'colsample_bytree': best['colsample_bytree'],
            'min_child_weight': int(best['min_child_weight']),
            'gamma': best['gamma'],
            'reg_alpha': best['reg_alpha'],
            'reg_lambda': best['reg_lambda'],
            'objective': 'reg:squarederror',
            'eval_metric': 'rmse',
            'random_state': self.random_state
        }
        print("\nBest parameters found:")
        print(self.best_params)
        return self.best_params

    def predict(self, X):
        dmatrix = xgb.DMatrix(X)
        return self.best_model.predict(dmatrix)

    def compute_shap_values(self, X):
        explainer = shap.TreeExplainer(self.best_model)
        dmatrix = xgb.DMatrix(X)
        return explainer.shap_values(dmatrix)

In [ ]:
start_time = time.time()

X_1 = np.concatenate((X, coords), axis=1)
xgb_trainer = XGBoostTrainer_1(n_splits=5, random_state=42, n_jobs=8)
final_model, oof_preds, oof_rmse, oof_r2, oof_shap = xgb_trainer.calcu_oof_and_shap(X_1, y, tune=True, max_evals=50)

end_time = time.time()
elapsed_time = end_time - start_time

print(elapsed_time)

## Model evaluation

In [ ]:
print("RMSE: ", oof_rmse)
print("adj_R2: ", adjusted_r2_score(oof_preds, y, 6))

### residual bootstrap (shap values)

In [ ]:
def boostrap_shap(y_pred):
    n =2500
    shap_bootstrap_list = []
    err = y - y_pred
    
    for i in tqdm(range(25)):
        
        random_sample_index = np.random.choice(np.arange(2500), size=2500, replace=True)
        
        y_sample = y_pred + err[random_sample_index]
        print(y_sample.shape)
        
        xgb_trainer = XGBoostTrainer_1(n_splits=5, random_state=42, n_jobs=8)
        final_model, oof_preds, oof_rmse, oof_r2, oof_shap = xgb_trainer.calcu_oof_and_shap(X_1, y_sample, tune=True, max_evals=50)

        shap_bootstrap_list.append(oof_shap)

    return np.concatenate(shap_bootstrap_list, axis=0)

In [ ]:
%%time

np.random.seed(333)
oof_preds_1 = oof_preds.reshape(2500, 1)

shap_bootstrap_list= boostrap_shap(oof_preds_1)

## Plotting

### 95% confidence level

In [ ]:
for i in range(6):
    plt.figure(figsize=(10, 7))
    
    # 为了绘制平滑的置信区间，需要按 X 值排序
    # 获取当前特征的数据
    x_vals = X[:, i]
    pred_vals = oof_shap[:, i]
    true_vals = fs[:, i]
    lower_vals = shap_ci_lower_simple[:, i]  # 假设您有对应的 lower 数组
    upper_vals = shap_ci_upper_simple[:, i]  # 假设您有对应的 upper 数组
    
    # 按 X 值排序（确保置信区间绘制正确）
    sort_idx = np.argsort(x_vals)
    x_sorted = x_vals[sort_idx]
    pred_sorted = pred_vals[sort_idx]
    lower_sorted = lower_vals[sort_idx]
    upper_sorted = upper_vals[sort_idx]
    
    # 绘制置信区间（半透明填充）
    plt.fill_between(x_sorted, lower_sorted, upper_sorted, 
                     color='k', alpha=0.2, label='95% Confidence Interval')
    
    # 绘制预测值散点
    plt.scatter(x_vals, pred_vals,
                c='none', edgecolors='firebrick',
                alpha=0.6, s=18, label='XGB curve')
    
    # 绘制真实值散点
    plt.scatter(x_vals, true_vals, 
                c='k', s=2.5, alpha=0.9, label='Actual curve')
    
    # 可选：绘制预测值的连线（显示趋势）
    # plt.plot(x_sorted, pred_sorted, color='firebrick', alpha=0.4, linewidth=1)
    
    # 计算并显示指标
    metric_val = calculate_metric(pred_vals, true_vals, chosen_metric)
    plt.text(locs_X[i], locs_Y[i], f'{chosen_metric}: {metric_val:.3f}',
             fontdict={'size': 24})
    
    plt.legend(loc=locs[i], fontsize = 20)
    plt.ylabel(fr'$\phi_{i+1} (X_{i+1})$')
    plt.xlabel(fr'$\mathrm{{X}}_{i+1}$')
    
    plt.savefig(simu_path + '\\' + fr'xgb_X{i+1}_nl.jpg', dpi=300, bbox_inches='tight')
    plt.close()

### Spatial

In [ ]:
for i in range(6):

    fig = plt.figure(figsize = (8, 8))

    pesudo = calculate_tikhonov_local_coefficients(oof_shap[:, i], X[:, i])
    
    plot_1(pesudo, fr'XGB: $f_{i+1} $', vmin = vmins[i], vmax = vmaxs[i])
    
    expected = fs[:, i] / X[:, i].reshape(-1)
    # cosine_sim = calculate_metric(expected, pesudo, chosen_metric)
    # predicted = moving_wind(oof_shap[:, i], X[:, i]).reshape(-1)
    cosine_sim = np.dot(pesudo, expected)/(np.linalg.norm(pesudo) * np.linalg.norm(expected))
    
    print(f'the cosine similarity index for x_{i+1} is {cosine_sim}')

    plt.savefig(simu_path + '\\' + fr'xgb_X{i+1}_s.jpg', 
                dpi = 300)

    plt.close()

# GWXGB

In [ ]:
gwxgb_data = pd.read_csv(result_path  + '\\' + r'gwx_full_analysis_results.csv')

y = np.array(gwxgb_data['y_true']).reshape(-1, 1)
y_hat = np.array(gwxgb_data['prediction']).reshape(-1, 1)
X = np.array(gwxgb_data[['x1', 'x2', 'x3', 'x4', 'x5', 'x6']]).reshape(2500, 6)
fs = np.array(gwxgb_data[['f1_true', 'f2_true', 'f3_true', 'f4_true', 'f5_true', 'f6_true']]).reshape(2500, 6)
fs_hat = np.array(gwxgb_data[['shap_x1', 'shap_x2', 'shap_x3', 'shap_x4', 'shap_x5', 'shap_x6']]).reshape(2500, 6)

fs_hat_ci = pd.read_csv(r'E:\PaperCode\MGWXGB\Revision\Results\CI\gwxgb_ci_results.csv')
fs_lower = np.array(fs_hat_ci[['x1_shap_ci_lower', 'x2_shap_ci_lower', 'x3_shap_ci_lower', 'x4_shap_ci_lower', 'x5_shap_ci_lower', 'x6_shap_ci_lower']]).reshape(2500, 6)
fs_upper = np.array(fs_hat_ci[['x1_shap_ci_upper', 'x2_shap_ci_upper', 'x3_shap_ci_upper', 'x4_shap_ci_upper', 'x5_shap_ci_upper', 'x6_shap_ci_upper']]).reshape(2500, 6)


## Model evaluation

In [ ]:
print("RMSE: ", np.sqrt(mean_squared_error(y, y_hat)))
print("adj_R2: ", adjusted_r2_score(y, y_hat, 6))
      

## Plotting

### 95% confidence level

In [ ]:
## global relationships
locs_X = [0.5, 0.5, 0.5, -0.5, 0.5, 0.5]
locs_Y = [-3.2, -2, -4, -4, -3, -3]
locs = ['upper left', 'upper left', 'upper left','upper center', 'upper left', 'upper left']

chosen_metric = 'RMSE'

for i in range(6):
    plt.figure(figsize=(10, 7))
    
    # 为了绘制平滑的置信区间，需要按 X 值排序
    # 获取当前特征的数据
    x_vals = X[:, i]
    pred_vals = fs_hat[:, i]
    true_vals = fs[:, i]
    lower_vals = fs_lower[:, i]  # 假设您有对应的 lower 数组
    upper_vals = fs_upper[:, i]  # 假设您有对应的 upper 数组
    
    # 按 X 值排序（确保置信区间绘制正确）
    sort_idx = np.argsort(x_vals)
    x_sorted = x_vals[sort_idx]
    pred_sorted = pred_vals[sort_idx]
    lower_sorted = lower_vals[sort_idx]
    upper_sorted = upper_vals[sort_idx]
    
    # 绘制置信区间（半透明填充）
    plt.fill_between(x_sorted, lower_sorted, upper_sorted, 
                     color='k', alpha=0.2, label='95% Confidence Interval')
    
    # 绘制预测值散点
    plt.scatter(x_vals, pred_vals,
                c='none', edgecolors='firebrick',
                alpha=0.6, s=18, label='GWXGB curve')
    
    # 绘制真实值散点
    plt.scatter(x_vals, true_vals, 
                c='k', s=2.5, alpha=0.9, label='Actual curve')
    
    # 可选：绘制预测值的连线（显示趋势）
    # plt.plot(x_sorted, pred_sorted, color='firebrick', alpha=0.4, linewidth=1)
    
    # 计算并显示指标
    metric_val = calculate_metric(pred_vals, true_vals, chosen_metric)
    plt.text(locs_X[i], locs_Y[i], f'{chosen_metric}: {metric_val:.3f}',
             fontdict={'size': 24})
    
    plt.legend(loc=locs[i], fontsize = 20)
    plt.ylabel(fr'$\phi_{i+1} (X_{i+1})$')
    plt.xlabel(fr'$\mathrm{{X}}_{i+1}$')
    
    plt.savefig(simu_path + '\\' + fr'gwx_X{i+1}_nl.jpg', dpi=300, bbox_inches='tight')
    plt.close()

### Spatial

In [ ]:

for i in range(6):

    fig = plt.figure(figsize = (8, 8))

    pesudo = calculate_tikhonov_local_coefficients(fs_hat[:, i], X[:, i])
    plot_1(pesudo, fr'GWXGB: $f_{i+1} $', vmin = vmins[i], vmax = vmaxs[i])
    
    expected = fs[:, i] / X[:, i].reshape(-1)
    cosine_sim = np.dot(pesudo, expected)/(np.linalg.norm(pesudo) * np.linalg.norm(expected))
    
    print(f'the cosine similarity index for x_{i+1} is {cosine_sim}')

    plt.savefig(simu_path + '\\' + fr'gwx_X{i+1}_s.jpg', 
                dpi = 300)

    plt.close()

# GXGB

In [ ]:
gxgb_data = pd.read_csv(result_path  + '\\' + r'gx_full_analysis_results.csv')

y = np.array(gxgb_data['y_true']).reshape(-1, 1)
y_hat = np.array(gxgb_data['prediction']).reshape(-1, 1)
X = np.array(gxgb_data[['x1', 'x2', 'x3', 'x4', 'x5', 'x6']]).reshape(2500, 6)
fs = np.array(gxgb_data[['f1_true', 'f2_true', 'f3_true', 'f4_true', 'f5_true', 'f6_true']]).reshape(2500, 6)
fs_hat = np.array(gxgb_data[['shap_x1', 'shap_x2', 'shap_x3', 'shap_x4', 'shap_x5', 'shap_x6']]).reshape(2500, 6)

In [ ]:
fs_hat_ci = pd.read_csv(r'E:\PaperCode\MGWXGB\Revision\Results\CI\gxgb_ci_results.csv')
fs_lower = np.array(fs_hat_ci[['x1_shap_ci_lower', 'x2_shap_ci_lower', 'x3_shap_ci_lower', 'x4_shap_ci_lower', 'x5_shap_ci_lower', 'x6_shap_ci_lower']]).reshape(2500, 6)
fs_upper = np.array(fs_hat_ci[['x1_shap_ci_upper', 'x2_shap_ci_upper', 'x3_shap_ci_upper', 'x4_shap_ci_upper', 'x5_shap_ci_upper', 'x6_shap_ci_upper']]).reshape(2500, 6)


## Model evaluation

In [ ]:
print("RMSE: ", np.sqrt(mean_squared_error(y, y_hat)))
print("adj_R2: ", adjusted_r2_score(y, y_hat, 6))
      

## Plotting

### Non-linear

In [ ]:
## global relationships
locs_X = [0.5, 0.5, 0.5, -0.5, 0.5, 0.5]
locs_Y = [-3.2, -2, -4, -4, -3, -3]
locs = ['upper left', 'upper left', 'upper left','upper center', 'upper left', 'upper left']

chosen_metric = 'RMSE'

for i in range(6):
    plt.figure(figsize=(10, 7))
    
    # 为了绘制平滑的置信区间，需要按 X 值排序
    # 获取当前特征的数据
    x_vals = X[:, i]
    pred_vals = fs_hat[:, i]
    true_vals = fs[:, i]
    lower_vals = fs_lower[:, i]  # 假设您有对应的 lower 数组
    upper_vals = fs_upper[:, i]  # 假设您有对应的 upper 数组
    
    # 按 X 值排序（确保置信区间绘制正确）
    sort_idx = np.argsort(x_vals)
    x_sorted = x_vals[sort_idx]
    pred_sorted = pred_vals[sort_idx]
    lower_sorted = lower_vals[sort_idx]
    upper_sorted = upper_vals[sort_idx]
    
    # 绘制置信区间（半透明填充）
    plt.fill_between(x_sorted, lower_sorted, upper_sorted, 
                     color='k', alpha=0.2, label='95% Confidence Interval')
    
    # 绘制预测值散点
    plt.scatter(x_vals, pred_vals,
                c='none', edgecolors='firebrick',
                alpha=0.6, s=18, label='GXGB curve')
    
    # 绘制真实值散点
    plt.scatter(x_vals, true_vals, 
                c='k', s=2.5, alpha=0.9, label='Actual curve')
    
    # 可选：绘制预测值的连线（显示趋势）
    # plt.plot(x_sorted, pred_sorted, color='firebrick', alpha=0.4, linewidth=1)
    
    # 计算并显示指标
    metric_val = calculate_metric(pred_vals, true_vals, chosen_metric)
    plt.text(locs_X[i], locs_Y[i], f'{chosen_metric}: {metric_val:.3f}',
             fontdict={'size': 24})
    
    plt.legend(loc=locs[i], fontsize = 20)
    plt.ylabel(fr'$\phi_{i+1} (X_{i+1})$')
    plt.xlabel(fr'$\mathrm{{X}}_{i+1}$')
    
    plt.savefig(simu_path + '\\' + fr'gx_X{i+1}_nl.jpg', dpi=300, bbox_inches='tight')
    plt.close()

### Spatial

In [ ]:
for i in range(6):

    fig = plt.figure(figsize = (8, 8))

    plot_1(fs_hat[:, i]/ X[:, i], fr'GXGB: $f_{i+1} $', vmin = vmins[i], vmax = vmaxs[i])
    
    pesudo = calculate_tikhonov_local_coefficients(fs_hat[:, i], X[:, i])
    expected = fs[:, i] / X[:, i].reshape(-1)
    cosine_sim = np.dot(pesudo, expected)/(np.linalg.norm(pesudo) * np.linalg.norm(expected))
    
    print(f'the cosine similarity index for x_{i+1} is {cosine_sim}')

    plt.savefig(simu_path + '\\' + fr'gx_X{i+1}_s.jpg', 
                dpi = 300)

    plt.close()

# MGWX

In [ ]:
mgwx_data = pd.read_csv(result_path  + '\\' + r'task2_mgwx_global_results.csv')

y = np.array(mgwx_data['y_true']).reshape(-1, 1)
y_hat = np.array(mgwx_data['y_hat']).reshape(-1, 1)
X = np.array(mgwx_data[['x1', 'x2', 'x3', 'x4', 'x5', 'x6']]).reshape(2500, 6)
fs = np.array(mgwx_data[['f1_true', 'f2_true', 'f3_true', 'f4_true', 'f5_true', 'f6_true']]).reshape(2500, 6)
fs_hat = np.array(mgwx_data[['f1_pred', 'f2_pred', 'f3_pred', 'f4_pred', 'f5_pred', 'f6_pred']]).reshape(2500, 6)

fs_hat_ci = fs_hat_ci = pd.read_csv(result_path  + '\\' + r'task2_mgwx_CI_global_CI.csv')
fs_lower = np.array(fs_hat_ci[['f1_lower_95', 'f2_lower_95', 'f3_lower_95', 'f4_lower_95', 'f5_lower_95', 'f6_lower_95']]).reshape(2500, 6)
fs_upper = np.array(fs_hat_ci[['f1_upper_95', 'f2_upper_95', 'f3_upper_95', 'f4_upper_95', 'f5_upper_95', 'f6_upper_95']]).reshape(2500, 6)


## Model evaluation

In [ ]:
print("RMSE: ", np.sqrt(mean_squared_error(y, y_hat)))
print("adj_R2: ", adjusted_r2_score(y, y_hat, 6))
      

## plotting

### %95 confidence level

In [ ]:
## global relationships
locs_X = [0.5, 0.5, 0.5, -0.5, 0.5, 0.5]
locs_Y = [-3.2, -2, -4, -4, -3, -3]
locs = ['upper left', 'upper left', 'upper left','upper center', 'upper left', 'upper left']

chosen_metric = 'RMSE'

for i in range(6):
    plt.figure(figsize=(10, 7))
    
    # 为了绘制平滑的置信区间，需要按 X 值排序
    # 获取当前特征的数据
    x_vals = X[:, i]
    pred_vals = fs_hat[:, i]
    true_vals = fs[:, i]
    lower_vals = fs_lower[:, i]  # 假设您有对应的 lower 数组
    upper_vals = fs_upper[:, i]  # 假设您有对应的 upper 数组
    
    # 按 X 值排序（确保置信区间绘制正确）
    sort_idx = np.argsort(x_vals)
    x_sorted = x_vals[sort_idx]
    pred_sorted = pred_vals[sort_idx]
    lower_sorted = lower_vals[sort_idx]
    upper_sorted = upper_vals[sort_idx]
    
    # 绘制置信区间（半透明填充）
    plt.fill_between(x_sorted, lower_sorted, upper_sorted, 
                     color='k', alpha=0.2, label='95% Confidence Interval')
    
    # 绘制预测值散点
    plt.scatter(x_vals, pred_vals,
                c='none', edgecolors='firebrick',
                alpha=0.6, s=18, label='MGWX curve')
    
    # 绘制真实值散点
    plt.scatter(x_vals, true_vals, 
                c='k', s=2.5, alpha=0.9, label='Actual curve')
    
    # 可选：绘制预测值的连线（显示趋势）
    # plt.plot(x_sorted, pred_sorted, color='firebrick', alpha=0.4, linewidth=1)
    
    # 计算并显示指标
    metric_val = calculate_metric(pred_vals, true_vals, chosen_metric)
    plt.text(locs_X[i], locs_Y[i], f'{chosen_metric}: {metric_val:.3f}',
             fontdict={'size': 24})
    
    plt.legend(loc=locs[i], fontsize = 20)
    plt.ylabel(fr'$\phi_{i+1} (X_{i+1})$')
    plt.xlabel(fr'$\mathrm{{X}}_{i+1}$')
    
    plt.savefig(simu_path + '\\' + fr'mgwx_X{i+1}_nl.jpg', dpi=300, bbox_inches='tight')
    plt.close()

### Spatial

In [ ]:
vmins = [-0.682457051890915, 3.189114018598005e-05, 0.0, -2.9987223024571503, 7.360470597936544e-08, 1.8]
vmaxs = [3.14158952787694, 3.141592563325639, 2.997501561632397, 2.9990105934556786, 2.2489935724983514, 2.2]

for i in range(6):

    fig = plt.figure(figsize = (8, 8))
    
    expected = fs[:, i] / X[:, i].reshape(-1)
    pesudo = calculate_tikhonov_local_coefficients(fs_hat[:, i], X[:, i])
    plot_1(pesudo, fr'MGWX: $f_{i+1} $', vmin = vmins[i], vmax = vmaxs[i])
    
    cosine_sim = np.dot(pesudo, expected)/(np.linalg.norm(pesudo) * np.linalg.norm(expected))
    
    print(f'the cosine similarity index for x_{i+1} is {cosine_sim}')

    plt.savefig(simu_path + '\\' + fr'mgwx_X{i+1}_s.jpg', 
                dpi = 300)

    plt.close()

# MGX

In [ ]:
mgwx_data = pd.read_csv(result_path  + '\\' + r'task2_mgx_global_results.csv')

y = np.array(mgwx_data['y_true']).reshape(-1, 1)
y_hat = np.array(mgwx_data['y_hat']).reshape(-1, 1)
X = np.array(mgwx_data[['x1', 'x2', 'x3', 'x4', 'x5', 'x6']]).reshape(2500, 6)
fs = np.array(mgwx_data[['f1_true', 'f2_true', 'f3_true', 'f4_true', 'f5_true', 'f6_true']]).reshape(2500, 6)
fs_hat = np.array(mgwx_data[['f1_pred', 'f2_pred', 'f3_pred', 'f4_pred', 'f5_pred', 'f6_pred']]).reshape(2500, 6)

fs_hat_ci = pd.read_csv(result_path  + '\\' + r'task2_mgx_CI_global_CI.csv')
fs_lower = np.array(fs_hat_ci[['f1_lower_95', 'f2_lower_95', 'f3_lower_95', 'f4_lower_95', 'f5_lower_95', 'f6_lower_95']]).reshape(2500, 6)
fs_upper = np.array(fs_hat_ci[['f1_upper_95', 'f2_upper_95', 'f3_upper_95', 'f4_upper_95', 'f5_upper_95', 'f6_upper_95']]).reshape(2500, 6)

## Model evaluation

In [ ]:
print("RMSE: ", np.sqrt(mean_squared_error(y, y_hat)))
print("adj_R2: ", adjusted_r2_score(y, y_hat, 6))
      

## Plotting

### Non-linear

In [ ]:
## global relationships
locs_X = [0.5, 0.5, 0.5, -0.5, 0.5, 0.5]
locs_Y = [-3.2, -2, -4, -4, -3, -3]
locs = ['upper left', 'upper left', 'upper left','upper center', 'upper left', 'upper left']

chosen_metric = 'RMSE'

for i in range(6):
    plt.figure(figsize=(10, 7))
    
    # 为了绘制平滑的置信区间，需要按 X 值排序
    # 获取当前特征的数据
    x_vals = X[:, i]
    pred_vals = fs_hat[:, i]
    true_vals = fs[:, i]
    lower_vals = fs_lower[:, i]  # 假设您有对应的 lower 数组
    upper_vals = fs_upper[:, i]  # 假设您有对应的 upper 数组
    
    # 按 X 值排序（确保置信区间绘制正确）
    sort_idx = np.argsort(x_vals)
    x_sorted = x_vals[sort_idx]
    pred_sorted = pred_vals[sort_idx]
    lower_sorted = lower_vals[sort_idx]
    upper_sorted = upper_vals[sort_idx]
    
    # 绘制置信区间（半透明填充）
    plt.fill_between(x_sorted, lower_sorted, upper_sorted, 
                     color='k', alpha=0.2, label='95% Confidence Interval')
    
    # 绘制预测值散点
    plt.scatter(x_vals, pred_vals,
                c='none', edgecolors='firebrick',
                alpha=0.6, s=18, label='MGX curve')
    
    # 绘制真实值散点
    plt.scatter(x_vals, true_vals, 
                c='k', s=2.5, alpha=0.9, label='Actual curve')
    
    # 可选：绘制预测值的连线（显示趋势）
    # plt.plot(x_sorted, pred_sorted, color='firebrick', alpha=0.4, linewidth=1)
    
    # 计算并显示指标
    metric_val = calculate_metric(pred_vals, true_vals, chosen_metric)
    plt.text(locs_X[i], locs_Y[i], f'{chosen_metric}: {metric_val:.3f}',
             fontdict={'size': 24})
    
    plt.legend(loc=locs[i], fontsize = 20)
    plt.ylabel(fr'$\phi_{i+1} (X_{i+1})$')
    plt.xlabel(fr'$\mathrm{{X}}_{i+1}$')
    
    plt.savefig(simu_path + '\\' + fr'mgx_X{i+1}_nl.jpg', dpi=300, bbox_inches='tight')
    plt.close()

### Spatial

In [ ]:
locs_X = [0.5, 0.5, 0.5, 0.5, 0.5, 0.8]
locs_Y = [-3.2, -2, -4, -4, -1.2, -2.6]

for i in range(6):

    fig = plt.figure(figsize = (8, 8))

    plot_1(fs_hat[:, i]/ X[:, i], fr'MGX: $f_{i+1} $', vmin = vmins[i], vmax = vmaxs[i])
    
    expected = fs[:, i] / X[:, i].reshape(-1)
    pesudo = calculate_tikhonov_local_coefficients(fs_hat[:, i], X[:, i])
    # predicted = moving_wind(fs_hat[:, i], X[:, i]).reshape(-1)
    cosine_sim = np.dot(predicted, expected)/(np.linalg.norm(predicted) * np.linalg.norm(expected))
    
    print(f'the cosine similarity index for x_{i+1} is {cosine_sim}')

    # plt.savefig(simu_path + '\\' + fr'mgx_X{i+1}_s.jpg', 
    #             dpi = 300)

    plt.close()

# HMGWX

In [ ]:
new_data = pd.read_csv(result_path  + '\\' + r'task2_hmgwx_global_results.csv')

y = np.array(new_data['y_true']).reshape(-1, 1)
y_hat = np.array(new_data['y_hat']).reshape(-1, 1)
X = np.array(new_data[['x1', 'x2', 'x3', 'x4', 'x5', 'x6']]).reshape(2500, 6)
fs = np.array(new_data[['f1_true', 'f2_true', 'f3_true', 'f4_true', 'f5_true', 'f6_true']]).reshape(2500, 6)
fs_hat = np.array(new_data[['f1_pred', 'f2_pred', 'f3_pred', 'f4_pred', 'f5_pred', 'f6_pred']]).reshape(2500, 6)

fs_hat_ci = pd.read_csv(result_path  + '\\' + r'task2_hmgwx_CI_global_CI.csv')
fs_lower = np.array(fs_hat_ci[['f1_lower_95', 'f2_lower_95', 'f3_lower_95', 'f4_lower_95', 'f5_lower_95', 'f6_lower_95']]).reshape(2500, 6)
fs_upper = np.array(fs_hat_ci[['f1_upper_95', 'f2_upper_95', 'f3_upper_95', 'f4_upper_95', 'f5_upper_95', 'f6_upper_95']]).reshape(2500, 6)


## Model evaluation

In [ ]:
print("RMSE: ", np.sqrt(mean_squared_error(y, y_hat)))

print("adj_R2: ", adjusted_r2_score(y, y_hat, 6))
      

## Plotting

### 95% confidence level

In [ ]:
## global relationships
locs_X = [0.5, 0.5, 0.5, -0.5, 0.5, 0.5]
locs_Y = [-3.2, -2, -4, -4, -3, -3]
locs = ['upper left', 'upper left', 'upper left','upper center', 'upper left', 'upper left']

chosen_metric = 'RMSE'

for i in range(6):
    plt.figure(figsize=(10, 7))
    
    # 为了绘制平滑的置信区间，需要按 X 值排序
    # 获取当前特征的数据
    x_vals = X[:, i]
    pred_vals = fs_hat[:, i]
    true_vals = fs[:, i]
    lower_vals = fs_lower[:, i]  # 假设您有对应的 lower 数组
    upper_vals = fs_upper[:, i]  # 假设您有对应的 upper 数组
    
    # 按 X 值排序（确保置信区间绘制正确）
    sort_idx = np.argsort(x_vals)
    x_sorted = x_vals[sort_idx]
    pred_sorted = pred_vals[sort_idx]
    lower_sorted = lower_vals[sort_idx]
    upper_sorted = upper_vals[sort_idx]
    
    # 绘制置信区间（半透明填充）
    plt.fill_between(x_sorted, lower_sorted, upper_sorted, 
                     color='k', alpha=0.2, label='95% Confidence Interval')
    
    # 绘制预测值散点
    plt.scatter(x_vals, pred_vals,
                c='none', edgecolors='firebrick',
                alpha=0.6, s=18, label='HMGWX curve')
    
    # 绘制真实值散点
    plt.scatter(x_vals, true_vals, 
                c='k', s=2.5, alpha=0.9, label='Actual curve')
    
    # 可选：绘制预测值的连线（显示趋势）
    # plt.plot(x_sorted, pred_sorted, color='firebrick', alpha=0.4, linewidth=1)
    
    # 计算并显示指标
    metric_val = calculate_metric(pred_vals, true_vals, chosen_metric)
    plt.text(locs_X[i], locs_Y[i], f'{chosen_metric}: {metric_val:.3f}',
             fontdict={'size': 24})
    
    plt.legend(loc=locs[i], fontsize = 20)
    plt.ylabel(fr'$\phi_{i+1} (X_{i+1})$')
    plt.xlabel(fr'$\mathrm{{X}}_{i+1}$')
    
    plt.savefig(simu_path + '\\' + fr'hmgwx_X{i+1}_nl.jpg', dpi=300, bbox_inches='tight')
    plt.close()

### local

In [ ]:
def local_plot(x_vals, pred_vals, true_vals, lower_vals, upper_vals, simu_path):
    # 转换为 numpy 数组，确保可以使用位置索引
    i=1
    x_vals = np.asarray(x_vals)
    pred_vals = np.asarray(pred_vals)
    true_vals = np.asarray(true_vals)
    lower_vals = np.asarray(lower_vals)
    upper_vals = np.asarray(upper_vals)
    
    plt.figure(figsize=(8, 7))
    
    # 按 X 排序
    sort_idx = np.argsort(x_vals)
    x_sorted = x_vals[sort_idx]
    pred_sorted = pred_vals[sort_idx]
    lower_sorted = lower_vals[sort_idx]
    upper_sorted = upper_vals[sort_idx]
    
    # 绘制置信区间
    plt.fill_between(x_sorted, lower_sorted, upper_sorted, 
                     color='k', alpha=0.2, label='95% CI')
    
    # 预测值散点
    plt.scatter(x_vals, pred_vals,
                c='none', edgecolors='firebrick',
                alpha=0.6, s=18, label='HMGWX curve')
    
    # 真实值散点
    plt.scatter(x_vals, true_vals, 
                c='k', s=2.5, alpha=0.9, label='Actual curve')
    
    # 计算指标（假设 chosen_metric 和 locs_X, locs_Y 等已在外部定义）
    metric_val = calculate_metric(pred_vals, true_vals, 'RMSE')
    print(metric_val)
    plt.text(0.5, -2, f'RMSE: {metric_val:.3f}',
             fontdict={'size': 24})
    
    plt.legend(loc='upper left', fontsize=20)
    plt.ylabel(fr'$\phi_{i+1} (X_{i+1})$')
    plt.xlabel(fr'$\mathrm{{X}}_{i+1}$')
    
    plt.savefig(simu_path + '\\' + fr'hmgwx_X{i+1}_nl_2_1.jpg', dpi=300, bbox_inches='tight')
    plt.close()

In [568]:
local_results = pd.read_csv(r'E:\PaperCode\MGWXGB\Revision\Results\CI\HMGWX\task2_hmgwx_CI_local_CI.csv')

local_results.head()

,sub_model_index,original_index,prediction,feature_name,lower_95,upper_95
0,0,833,0.141069,x1,-0.506723,0.256387
1,0,783,0.551358,x1,-0.430107,0.559377
2,0,784,0.564606,x1,0.215635,1.035338
3,0,834,1.333505,x1,0.672868,1.353634
4,0,782,-1.304706,x1,-1.812314,-0.961616


In [569]:
bws_hmgwx = local_results.groupby(['feature_name'])['prediction'].count().reset_index()

bws_hmgwx['bw'] = bws_hmgwx['prediction'] / 250 
hmgwx_bws = bws_hmgwx['bw'].values

In [570]:
group_1 = local_results[local_results['feature_name'] == 'x1']
group_2 = local_results[local_results['feature_name'] == 'x2']

In [575]:
X_df = pd.DataFrame(X, columns=['x1', 'x2', 'x3', 'x4', 'x5', 'x6']).reset_index()
X_df.columns = ['original_index', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6']
X_df[['f1', 'f2', 'f3', 'f4', 'f5', 'f6']] = fs

group_1x = pd.merge(group_1, X_df.loc[:, ['original_index', 'x1', 'f1']], on = 'original_index')
group_2x = pd.merge(group_2, X_df.loc[:, ['original_index', 'x2', 'f2']], on = 'original_index')

In [ ]:
import random

random_number = random.randint(0, 250)
print(random_number)

index = group_2x[group_2x['sub_model_index'] == random_number].loc[:, 'original_index']
X_use = group_2x[group_2x['sub_model_index'] == random_number].loc[:, 'x2']
y_hat_use = group_2x[group_2x['sub_model_index'] == random_number].loc[:, 'prediction']
y_true_use = np.abs(X_use)*X_use
upper_use = group_2x[group_2x['sub_model_index'] == random_number].loc[:, 'upper_95']
lower_use = group_2x[group_2x['sub_model_index'] == random_number].loc[:, 'lower_95']

local_plot(X_use, y_hat_use, y_true_use, lower_use, upper_use, simu_path = simu_path)


In [ ]:
for i in range(1,3):
    plt.figure(figsize=(10, 7))
    
    # 为了绘制平滑的置信区间，需要按 X 值排序
    # 获取当前特征的数据
    x_vals = X[:, i]
    pred_vals = fs_hat[:, i]
    true_vals = fs[:, i]
    lower_vals = fs_lower[:, i]  # 假设您有对应的 lower 数组
    upper_vals = fs_upper[:, i]  # 假设您有对应的 upper 数组
    
    # 按 X 值排序（确保置信区间绘制正确）
    sort_idx = np.argsort(x_vals)
    x_sorted = x_vals[sort_idx]
    pred_sorted = pred_vals[sort_idx]
    lower_sorted = lower_vals[sort_idx]
    upper_sorted = upper_vals[sort_idx]
    
    # 绘制置信区间（半透明填充）
    plt.fill_between(x_sorted, lower_sorted, upper_sorted, 
                     color='k', alpha=0.2, label='95% Confidence Interval')
    
    # 绘制预测值散点
    plt.scatter(x_vals, pred_vals,
                c='none', edgecolors='firebrick',
                alpha=0.6, s=18, label='HMGWX curve')
    
    # 绘制真实值散点
    plt.scatter(x_vals, true_vals, 
                c='k', s=2.5, alpha=0.9, label='Actual curve')
    
    # 可选：绘制预测值的连线（显示趋势）
    # plt.plot(x_sorted, pred_sorted, color='firebrick', alpha=0.4, linewidth=1)
    
    # 计算并显示指标
    metric_val = calculate_metric(pred_vals, true_vals, chosen_metric)
    plt.text(locs_X[i], locs_Y[i], f'{chosen_metric}: {metric_val:.3f}',
             fontdict={'size': 24})
    
    plt.legend(loc=locs[i], fontsize = 20)
    plt.ylabel(fr'$\phi_{i+1} (X_{i+1})$')
    plt.xlabel(fr'$\mathrm{{X}}_{i+1}$')
    
    plt.savefig(simu_path + '\\' + fr'hmgwx_X{i+1}_nl.jpg', dpi=300, bbox_inches='tight')
    plt.close()

### spatial

In [ ]:
vmins = [-0.682457051890915, 3.189114018598005e-05, 0.0, -2.9987223024571503, 7.360470597936544e-08, 1.8]
vmaxs = [3.14158952787694, 3.141592563325639, 2.997501561632397, 2.9990105934556786, 2.2489935724983514, 2.2]

for i in range(6):

    fig = plt.figure(figsize = (8, 8))
    expected = fs[:, i] / X[:, i].reshape(-1)
    pesudo = calculate_tikhonov_local_coefficients(fs_hat[:, i], X[:, i])
    plot_1(fs_hat[:, i]/ X[:, i], fr'HMGWX: $f_{i+1} $', vmin = vmins[i], vmax = vmaxs[i])
    
    cosine_sim = np.dot(pesudo, expected)/(np.linalg.norm(pesudo) * np.linalg.norm(expected))
    
    print(f'the cosine similarity index for x_{i+1} is {cosine_sim}')

    plt.savefig(simu_path + '\\' + fr'hmgwx_X{i+1}_s.jpg', 
                dpi = 300)

    plt.close()